## Two step RAG method

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import openai
from transformers import AutoTokenizer
import os
import utils
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get OpenRouter API key from environment variables
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise ValueError("Please set OPENROUTER_API_KEY in your .env file or environment variables")

/opt/anaconda3/envs/litigation/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Loading and cleaning the data

In [3]:
few_shot_examples = 'groundtruth_classifications.xlsx'
text_data = 'full_data_filtered.csv'

data = pd.read_csv(text_data)
examples = pd.read_excel(few_shot_examples)

In [4]:
# Clean up the examples
examples = examples[examples['File name'].notna()]
examples = examples.drop(columns=['Note'])
examples['year'] = examples['File name'].str.extract(r'-(\d{2})-')
examples = examples.replace({'Yes': 1, 'No': 0})

/var/folders/br/4mkn2pts7yg8xb1_dfd6g01c0000gn/T/ipykernel_58556/2978246190.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  examples = examples.replace({'Yes': 1, 'No': 0})


In [5]:
litigation_examples = examples.drop(columns=['Climate', 'Litigation', 'General risk', 'Specific lawsuit(s)', 'File name'])
litigation_examples.rename(columns={'Paragraph': 'text', 'Company': 'company', 'Climate Litigation': 'climate_litigation'}, inplace=True)

In [6]:
data.rename(columns={'text': 'text', 'folder': 'company'}, inplace=True)
data.drop(columns=['folderfiletext'], inplace=True, errors='ignore')

In [7]:
print(f"Loaded {len(data)} documents and {len(litigation_examples)} ground truth examples")


Loaded 528 documents and 61 ground truth examples


In [8]:
#count how many different companies there are in 'data' 
companies = data['company'].unique()
print(f"Found {len(companies)} unique companies in the data")

Found 44 unique companies in the data


### Chunking the text

In [9]:
tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1")

In [10]:
print("Chunking documents...")
expanded_rows = []
for _, row in data.iterrows():
    expanded_rows.extend(utils.tokenize_and_chunk(row, tokenizer))
df = pd.DataFrame(expanded_rows)
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)

print("Chunking ground truth examples...")
groundtruth_expanded = []
for _, row in litigation_examples.iterrows():
    groundtruth_expanded.extend(utils.tokenize_and_chunk(row, tokenizer))
groundtruth_df = pd.DataFrame(groundtruth_expanded)

print(f"Created {len(df)} document chunks and {len(groundtruth_df)} ground truth chunks")

Chunking documents...


Token indices sequence length is longer than the specified maximum sequence length for this model (44218 > 8192). Running this sequence through the model will result in indexing errors


Chunking ground truth examples...
Created 143478 document chunks and 61 ground truth chunks


### Setting up embedding

In [11]:
print("Loading embedding model...")
embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1", trust_remote_code=True)

Loading embedding model...


<All keys matched successfully>
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [11]:
print("Encoding document embeddings...")
doc_embeddings = utils.encode_in_batches(df['text'].tolist(), embedding_model, batch_size=10)

print("Encoding ground truth embeddings...")
gt_embeddings = utils.encode_in_batches(groundtruth_df['text'].tolist(), embedding_model, batch_size=10)

Encoding document embeddings...


Embedding:   5%|▍         | 656/14348 [06:28<2:15:16,  1.69it/s]


KeyboardInterrupt: 

In [ ]:
doc_embeddings, valid_doc_idx = doc_embeddings
gt_embeddings, valid_gt_idx = gt_embeddings

In [ ]:
np.save("doc_embeddings.npy", doc_embeddings)
np.save("gt_embeddings.npy", gt_embeddings)

### Using the AI model

In [12]:
doc_embeddings = np.load("doc_embeddings.npy")
gt_embeddings = np.load("gt_embeddings.npy")

In [13]:
df["embedding"] = list(doc_embeddings)
groundtruth_df["embedding"] = list(gt_embeddings)

In [14]:
company_groups = df.groupby('company')

In [ ]:
client = openai.OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

In [16]:
import importlib 
utils = importlib.reload(utils)

In [17]:
start_company = "SUN"
start_year = 2019
started = False

for company, company_df in company_groups:
    if not started:
        if company < start_company:
            continue
        elif company == start_company:
            # Filter to only years >= start_year for the start_company
            company_df = company_df[company_df["year"] >= start_year]
            if company_df.empty:
                continue
            started = True
        else:
            started = True  # first company > start_company

    # Once started, process all years in this company_df
    for year, year_df in company_df.groupby("year"):
        nested_output_dir = os.path.join("META_rag_results", company, str(year))
        os.makedirs(nested_output_dir, exist_ok=True)

        results_df = utils.run_rag_classification_for_company(
            embedding_model=embedding_model,
            groundtruth_df=groundtruth_df,
            client=client,
            company_df=year_df.reset_index(drop=True),
            company_name=company,
            retrieval_k=100,
            example_k=5,
            start_index=0,
            output_dir=nested_output_dir
        )



🔍 Processing company: SUN with 207 chunks
Index(['company', 'year', 'text', 'embedding'], dtype='object')
Classifying 100 candidate chunks for SUN
Tokens - Prompt: 2091, Completion: 7, Total: 2098
[SUN][Year 2019] Chunk 1/100: Environmental Matters
Environmental Laws and Regul... -> climate_litigation: 1
Tokens - Prompt: 2429, Completion: 6, Total: 2435
[SUN][Year 2019] Chunk 2/100: However, the current administration under Presiden... -> climate_litigation: 0
Tokens - Prompt: 2454, Completion: 6, Total: 2460
[SUN][Year 2019] Chunk 3/100: Under the Clean Air Act and comparable state and l... -> climate_litigation: 0
Tokens - Prompt: 2027, Completion: 6, Total: 2033
[SUN][Year 2019] Chunk 4/100: To the extent third parties (including insurers) d... -> climate_litigation: 1
Tokens - Prompt: 2160, Completion: 7, Total: 2167
[SUN][Year 2019] Chunk 5/100: Costs associated with the investigation and remedi... -> climate_litigation: 0
Tokens - Prompt: 2229, Completion: 6, Total: 2235
[SUN][Y